In [1]:
# 데이터 처리 및 분석
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import ast

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # maxOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'
    
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

SB_DEEP_GREEN = '#1E3932'
SB_GREEN      = '#006241'
SB_LIGHT_GREEN = '#D4E9E2'
SB_GOLD       = '#CBA258'
SB_GREY       = '#A2AAAD'
SB_BLACK      = '#27251F'

plt.rcParams.update({
    'font.family': 'Malgun Gothic',
    'axes.unicode_minus': False,
    'text.color': SB_BLACK,
    'axes.labelcolor': SB_BLACK,
    'xtick.color': SB_BLACK,
    'ytick.color': SB_BLACK,
    'axes.spines.top': False,    
    'axes.spines.right': False, 
    'patch.edgecolor': 'none'    
})


sns.set_palette([SB_GREEN, SB_GOLD, SB_DEEP_GREEN, SB_LIGHT_GREEN, SB_GREY])

In [3]:
merge_df = pd.read_csv("../../Data/merged_df_260325.csv")

In [4]:
trans_df = pd.read_csv("../../Data/transactions_260325.csv")

In [5]:

# 2. 인포메이셔널 오퍼 제외 (순수 보상형 오퍼만 분석)
merge_filtered = merge_df[~merge_df['offer_label'].str.contains('informational', na=False)].copy()

# 3. '다중오퍼(multi)'로 완료된 결제 정보 가져오기
multi_trans = trans_df[trans_df['txn_offer_type'] == 'multi'][['person', 'time']].drop_duplicates()
multi_trans['is_multi_txn'] = True

# 4. 완료(completed) 이벤트 중 다중오퍼에 해당하는 건들 마킹
completions = merge_filtered[merge_filtered['event'] == 'completed'].copy()
completions = pd.merge(completions, multi_trans, on=['person', 'time'], how='left')

# 5. 각 오퍼 경로(person, offer_id, receive_seq)가 결국 '다중오퍼'로 끝났는지 태그 생성
multi_path_tags = completions[completions['is_multi_txn'] == True][['person', 'offer_id', 'receive_seq']].drop_duplicates()
multi_path_tags['final_funnel_cat'] = '다중오퍼'

# 6. 메인 데이터에 '최종 퍼널 카테고리' 부여
# 다중오퍼로 끝난 경로는 '다중오퍼'로, 나머지는 원래 'offer_label' 유지
merge_final = pd.merge(merge_filtered, multi_path_tags, on=['person', 'offer_id', 'receive_seq'], how='left')
merge_final['final_funnel_cat'] = merge_final['final_funnel_cat'].fillna(merge_final['offer_label'])

# 7. [Funnel 1단계] 발송수(Received) 집계
funnel_1_received = merge_final[merge_final['event'] == 'received']['final_funnel_cat'].value_counts().reset_index()
funnel_1_received.columns = ['카테고리', '발송수']

# 보기 좋게 정렬 (8종 + 다중오퍼)
order = ['bogo_1', 'bogo_2', 'bogo_3', 'bogo_4', 'discount_1', 'discount_2', 'discount_3', 'discount_4', '다중오퍼']
funnel_1_received['카테고리'] = pd.Categorical(funnel_1_received['카테고리'], categories=order, ordered=True)
funnel_1_received = funnel_1_received.sort_values('카테고리').reset_index(drop=True)

# 결과 확인
print("--- Funnel 1: 오퍼별 총 발송수 (Received) ---")
display(funnel_1_received)
print(f"\n총 발송 건수 합계: {funnel_1_received['발송수'].sum():,}건")

--- Funnel 1: 오퍼별 총 발송수 (Received) ---


,카테고리,발송수
0,bogo_1,7322
1,bogo_2,7320
2,bogo_3,7260
3,bogo_4,7268
4,discount_1,7249
5,discount_2,7351
6,discount_3,7258
7,discount_4,7323
8,다중오퍼,2691



총 발송 건수 합계: 61,042건
